# **🗃️ Data Loading**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import probplot, norm
from sklearn.metrics import mean_squared_error as mse

import warnings
warnings.filterwarnings('ignore')

## Load data and take a small look

In [ ]:
data = pd.read_csv('../Data/train_houses_iowa_original.csv')

print (data.columns)
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# sns.scatterplot(data, x = 'OverallQual', y = 'SalePrice')
sns.boxplot(data, x = 'OverallQual', y = 'SalePrice')

## Data Cleansing

In [ ]:
num_missing = data.isnull().sum().sort_values(ascending=False)
missing_percentage = (data.isnull().sum() / data.isnull().count()).sort_values(ascending=False)

num_missing.head(20)
missing = pd.concat([num_missing, missing_percentage], axis = 1, keys=['Total', 'Percentage'])
print(missing.head(20))

In [ ]:
drop_columns = missing[ missing['Percentage']>0.15 ]
print(drop_columns)

In [ ]:
data_cl = data.drop(data[(missing[ missing['Percentage']>0.15 ]).index], axis=1 )
data_cl.isnull().sum().sort_values(ascending=False).head(20)
# data_cl.head(10)

In [ ]:
data_cl.isnull().sum().sort_values(ascending=False).keys().tolist()

In [ ]:
for col in data_cl.isnull().sum().sort_values(ascending=False).keys().tolist():
  data_cl = data_cl.drop(data_cl.loc[data_cl[col].isnull()].index)
  print(col)

data_cl.isnull().sum().sort_values(ascending=False).min()
print (len(data_cl))

## Outliers

In [ ]:
data_cl.columns

In [ ]:
scaled_data_cl = StandardScaler().fit_transform(data_cl['SalePrice'].values.reshape(-1,1))

lower_bound = scaled_data_cl[scaled_data_cl[:, 0].argsort()][:10]
upper_bound = scaled_data_cl[scaled_data_cl[:, 0].argsort()][-10:]
print(lower_bound, upper_bound)

In [ ]:
sns.histplot(data_cl, x= 'SalePrice', kde=True, bins=len(np.arange(0,len(data_cl),10)))

## Normality test

In [ ]:
sns.distplot(data_cl['SalePrice'], fit = norm)
fig = plt.figure()
res = probplot(data_cl['SalePrice'], plot = plt)

In [ ]:
data_cl.info()

In [ ]:
data_cl = data_cl.select_dtypes(include = ['float64', 'int64'])
data_cl.info()

In [ ]:
data_cl.head()

In [ ]:
# Transformación de los datos:
data_cl_tf = data_cl.copy()

for col in data_cl.columns.tolist():
  # data_cl_tf[col] = np.log(data_cl[col])
  data_cl_tf[col].loc[data_cl_tf[col] != 0] = np.log(data_cl[col].loc[data_cl[col] != 0])
  # print (col)

In [ ]:
# Histograma y gráfico de probabilidad normal sobre los datos transformados:

sns.distplot(data_cl_tf['SalePrice'], fit = norm);
fig = plt.figure()
res = probplot(data_cl_tf['SalePrice'] , plot = plt)

In [ ]:
sns.distplot(data_cl['TotalBsmtSF'], fit = norm);
fig = plt.figure()
res = probplot(data_cl['TotalBsmtSF'], plot = plt)

In [ ]:
sns.distplot(data_cl_tf['TotalBsmtSF'], fit = norm);
fig = plt.figure()
res = probplot(data_cl_tf['TotalBsmtSF'], plot = plt)

In [ ]:
data_cl_tf.head()

In [ ]:
print(np.unique(data_cl_tf['GarageCars']))
print (data_cl_tf.columns)

In [ ]:
sns.pairplot(data_cl_tf, corner=True)

In [ ]:
data_cl_tf = data_cl_tf.drop(['Id'], axis=1)

In [ ]:
# Compute the correlation matrix
corr_matrix = data_cl_tf.corr()

# _, ax = plt.subplots(figsize=(30,30)) 
# sns.heatmap(corr_matrix, annot=True, cmap='hot', ax=ax)

# Plot the clustermap
sns.clustermap(corr_matrix, 
               annot=True,      # Add correlation values as text
               fmt=".2f",       # Format the annotations
               cmap='hot',  # Color map
               linewidths=.7,   # Space between cells
               figsize=(15,15),
               cbar_pos=(1.02, 0.2, 0.03, 0.4) # Position the color bar
              )
plt.show()

## Data Modeling

### First solution - Sale price prediction

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

In [ ]:
print (data_cl_tf.columns)
X = np.array(data_cl_tf.iloc[:, :-1])
y = np.array(data_cl_tf.iloc[:,  -1])

print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 1)

print (X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100)
gbr.fit (X_train, y_train)

In [ ]:
gbr_predictions = gbr.predict(X_test)
mse_score = mse(y_test, gbr_predictions)

print ("MSE: {0:.5f}".format(mse_score))

In [ ]:
lnr = LinearRegression()
lnr.fit(X_train, y_train)

In [ ]:
lnr_predictions = lnr.predict(X_test)
mse_score = mse(y_test, lnr_predictions)

print ("MSE: {0:.5f}".format(mse_score))

In [ ]:
_, axes = plt.subplots(1,2, figsize=(10,5))

var_idx = 2
name_var = data_cl_tf.columns[var_idx]

axes[0].scatter(X_test[:, var_idx], y_test, facecolors="none", edgecolors='k')
axes[0].scatter(X_test[:, var_idx], gbr_predictions, c='b', alpha=0.5, edgecolors='k')

axes[1].scatter(X_test[:, var_idx], y_test, facecolors="none", edgecolors='k')
axes[1].scatter(X_test[:, var_idx], lnr_predictions, c='r', alpha=0.5, edgecolors='k')

axes[0].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")
axes[1].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")

In [ ]:
_, axes = plt.subplots(1,2, figsize=(10,5))

var_idx = 2
name_var = data_cl_tf.columns[var_idx]

sns.boxplot(x=X_test[:, var_idx], y=y_test, ax=axes[0])
sns.boxplot(x=X_test[:, var_idx], y=y_test, ax=axes[1])
sns.boxplot(x=X_test[:, var_idx], y=lnr_predictions, ax=axes[0])
sns.boxplot(x=X_test[:, var_idx], y=gbr_predictions, ax=axes[1])

axes[0].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")
axes[1].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")

axes[0].set_title("LRG"), axes[1].set_title("GBR")


### Second solution - Quality estimation

In [ ]:
data_cl_tf2 = data_cl_tf.copy()

vals = np.unique(data_cl_tf2['OverallQual'])
for i, val in enumerate(vals): data_cl_tf2['OverallQual'].replace(val, i, inplace=True)

data_cl_tf2.head()

In [ ]:
print (data_cl_tf2.columns)
X = np.concatenate([data_cl_tf2.iloc[:, :2], data_cl_tf2.iloc[:, 4:-1]], axis=1)
y = np.array(data_cl_tf2.iloc[:, 2])

print(X.shape, y.shape)

In [ ]:
np.unique(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 1)

print (X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
lgr = LogisticRegression()
lgr.fit(X_train, y_train)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

svm = SVC(kernel='rbf', gamma=0.5)
svm.fit(X_train, y_train)

In [ ]:
p_lgr = lgr.predict(X_test)
p_knn = knn.predict(X_test)
p_svm = svm.predict(X_test)

print ("Predictions Logistic Regression")
print (classification_report(y_test, p_lgr))

print ("Predictions k-NN")
print (classification_report(y_test, p_knn))

print ("Predictions SVM")
print (classification_report(y_test, p_svm))

In [ ]:
np.unique(y_test,return_counts=True)

#### Resampling

In [ ]:
from sklearn.utils import resample

X_resampled, y_resampled = [], []

for i in range(len(np.unique(y_test))):
  # Upsample minority class
  Xr, yr = resample(X[y==i], y[y==i],
                  n_samples=79,    # to match majority class
                  random_state=1)  # reproducible results

  X_resampled.extend(Xr)
  y_resampled.extend(yr)

X_resampled = np.array(X_resampled)
y_resampled = np.array(y_resampled)

In [ ]:
np.unique(y_resampled,return_counts=True)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state = 1)

print (X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
lgr = LogisticRegression()
lgr.fit(X_train, y_train)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

svm = SVC(kernel='rbf', gamma=0.01, C=10)
svm.fit(X_train, y_train)

In [ ]:
p_lgr = lgr.predict(X_test)
p_knn = knn.predict(X_test)
p_svm = svm.predict(X_test)

print ("Predictions Logistic Regression")
print (classification_report(y_test, p_lgr))

print ("Predictions k-NN")
print (classification_report(y_test, p_knn))

print ("Predictions SVM")
print (classification_report(y_test, p_svm))

In [ ]:
_, axes = plt.subplots(1,4, figsize=(16,4))

var_idx = 1
name_var = data_cl_tf.columns[var_idx]

axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolors='k', cmap='Paired')
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=p_lgr, edgecolors='k', cmap='Paired')
axes[2].scatter(X_test[:, 0], X_test[:, 1], c=p_knn, edgecolors='k', cmap='Paired')
axes[3].scatter(X_test[:, 0], X_test[:, 1], c=p_svm, edgecolors='k', cmap='Paired')


axes[0].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")
axes[1].set_xlabel(name_var), axes[0].set_ylabel("SalePrice")